# NB_08 — Stage 7: Baseline Comparison (EasyOCR)

This notebook establishes a classical OCR baseline to compare against the
fine-tuned Qwen2.5-VL-7B adapter from NB_05/06.

**Why EasyOCR and not TrOCR?**
TrOCR (`microsoft/trocr-base-handwritten`) was the original plan, but it
failed entirely on Arabic: its tokenizer was built for Latin script and has
no meaningful representation of Arabic characters, and its vision encoder
was pre-trained on English handwriting. Fine-tuning for 5 epochs on 1,120
images cannot overcome that — the model produced repetitive nonsense
syllables regardless of input. This is documented as a challenge in the paper.

EasyOCR is a drop-in replacement with native Arabic support. Its models were
trained on Arabic text from the start, so it produces recognizable output
without any fine-tuning. This makes it the appropriate off-the-shelf baseline:
it represents what a practitioner would reach for before investing in a
custom fine-tuned model.

**No training required.** This notebook runs inference only.  
**GPU recommended** for speed but not required.

**Prerequisites:** NB_00 (data setup), NB_06 (Qwen eval results for comparison).

## Step 8.1 — Mount Drive and set paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT  = '/content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project'
TRAIN_FILE    = f'{PROJECT_ROOT}/data/train/train.jsonl'
EVAL_FILE     = f'{PROJECT_ROOT}/data/eval/eval.jsonl'
IMAGE_DIR     = f'{PROJECT_ROOT}/data/trocr_images'
TROCR_OUTPUT  = f'{PROJECT_ROOT}/models/trocr_baseline'
LOG_DIR       = f'{PROJECT_ROOT}/logs/stage7'

os.makedirs(f'{IMAGE_DIR}/train', exist_ok=True)
os.makedirs(f'{IMAGE_DIR}/eval',  exist_ok=True)
os.makedirs(TROCR_OUTPUT,         exist_ok=True)
os.makedirs(LOG_DIR,              exist_ok=True)

print(f'PROJECT_ROOT  : {PROJECT_ROOT}')
print(f'Image dir     : {IMAGE_DIR}')
print(f'TrOCR output  : {TROCR_OUTPUT}')

Mounted at /content/drive
PROJECT_ROOT  : /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project
Image dir     : /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project/data/trocr_images
TrOCR output  : /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project/models/trocr_baseline


## Step 8.2 — Install dependencies

EasyOCR and jiwer are the only requirements. EasyOCR will download its
Arabic model weights (~100 MB) on first initialization.

In [2]:
# ── Step 8.2: Install ─────────────────────────────────────────────────────
!pip install easyocr jiwer --quiet
print('✓ Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 32.4 MB/s eta 0:00:00
✓ Dependencies installed.


## Step 8.3 — Initialize EasyOCR Arabic reader

EasyOCR supports Arabic natively. We initialize it with `['ar']` to load
the Arabic recognition model. Setting `gpu=True` speeds up inference
significantly — each image takes ~2-3s on GPU vs ~15s on CPU.

In [3]:
# ── Step 8.3: Initialize EasyOCR Arabic reader ────────────────────────────
import easyocr
reader = easyocr.Reader(['ar'], gpu=True)
print('✓ EasyOCR Arabic reader initialized.')

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete✓ EasyOCR Arabic reader initialized.


## Step 8.4 — Extract eval images from JSONL

The eval JSONL stores images as base64-encoded PNG data URLs inside each
message. We decode them to disk as PNG files so EasyOCR can read them.
Images that already exist on disk are skipped, so this cell is safe to
re-run.

In [4]:
import json, base64, os
from PIL import Image
import io

eval_pairs = []

with open(EVAL_FILE) as f:
    for i, line in enumerate(f):
        sample        = json.loads(line)
        image_b64     = None
        transcription = None

        for msg in sample['messages']:
            if msg['role'] == 'user':
                content = msg['content']
                if isinstance(content, list):
                    for item in content:
                        if isinstance(item, dict) and item.get('type') == 'image_url':
                            image_b64 = item['image_url']['url'].split(',', 1)[1]
            elif msg['role'] == 'assistant':
                try:
                    transcription = json.loads(msg['content']).get('transcription', '').strip()
                except Exception:
                    transcription = msg['content'].strip()

        if image_b64 and transcription:
            img_path = f'{IMAGE_DIR}/eval_{i:04d}.png'
            if not os.path.exists(img_path):
                img = Image.open(io.BytesIO(base64.b64decode(image_b64))).convert('RGB')
                img.save(img_path)
            eval_pairs.append((img_path, transcription))

print(f'✓ {len(eval_pairs)} eval image/transcription pairs ready.')

✓ 280 eval image/transcription pairs ready.


## Step 8.5 — Run EasyOCR on the eval set

We run EasyOCR on all 280 eval images and compute CER and WER against the
GPT reference transcriptions — the same reference used to evaluate Qwen in
NB_06. Using the same reference makes the comparison direct and fair.

`paragraph=True` tells EasyOCR to merge detected text regions into a single
string per image, which matches how our JSONL stores transcriptions.

Expected runtime: 5-8 minutes on GPU.

In [5]:
from jiwer import cer, wer
import json, os

predictions    = []
references     = []
sample_outputs = []

print('Running EasyOCR on 280 eval samples...')
for i, (img_path, ref_text) in enumerate(eval_pairs):
    try:
        result    = reader.readtext(img_path, detail=0, paragraph=True)
        pred_text = ' '.join(result).strip()
    except Exception:
        pred_text = ''

    predictions.append(pred_text)
    references.append(ref_text)

    if i < 10:
        sample_outputs.append({'reference': ref_text, 'predicted': pred_text})

    if (i + 1) % 50 == 0:
        print(f'  Processed {i+1}/280...')

pairs       = [(p, r) for p, r in zip(predictions, references) if r.strip()]
preds, refs = zip(*pairs)
avg_cer     = cer(list(refs), list(preds))
avg_wer     = wer(list(refs), list(preds))

print(f'\n=== EasyOCR Baseline Results ===')
print(f'CER            : {avg_cer:.4f}')
print(f'WER            : {avg_wer:.4f}')
print(f'Samples evaluated: {len(pairs)}')

Running EasyOCR on 280 eval samples...
  Processed 50/280...
  Processed 100/280...
  Processed 150/280...
  Processed 200/280...
  Processed 250/280...

=== EasyOCR Baseline Results ===
CER            : 0.4764
WER            : 1.0582
Samples evaluated: 280


## Step 8.6 — Save results

Results are saved in the same schema as the other stage logs so the final
summary notebook can load them uniformly.

In [8]:
import json, os

LOG_DIR_7 = f'{PROJECT_ROOT}/logs/stage7'
os.makedirs(LOG_DIR_7, exist_ok=True)

results = {
    'model':            'EasyOCR (Arabic)',
    'architecture':     'CRAFT detector + CRNN recognizer',
    'num_eval_samples': len(pairs),
    'cer':              round(avg_cer, 4),
    'wer':              round(avg_wer, 4),
    'sample_outputs':   sample_outputs,
    'note': (
        'Off-the-shelf Arabic OCR baseline. No fine-tuning performed. '
        'TrOCR (microsoft/trocr-base-handwritten) was attempted first but '
        'discarded — its English-only tokenizer and vision encoder are '
        'incompatible with Arabic script, producing nonsense output.'
    ),
}

with open(f'{LOG_DIR_7}/easyocr_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f'✓ Results saved to {LOG_DIR_7}/easyocr_results.json')

✓ Results saved to /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project/logs/stage7/easyocr_results.json


## Step 8.7 — Compare EasyOCR against Qwen2.5-VL-7B + LoRA

The table below puts the baseline directly alongside the fine-tuned Qwen
adapter evaluated in NB_06. Both are measured against the same GPT
reference transcriptions on the same 280-sample eval set.

A lower CER/WER is better. Values above 1.0 indicate the model is
generating more characters than the reference on average (over-generation).

In [9]:
import json

qwen_path = f'{PROJECT_ROOT}/logs/run-2/eval_results_checkpoint-1120.json'
with open(qwen_path) as f:
    qwen = json.load(f)

print('=== Stage 7: Baseline Comparison ===\n')
print(f'{"Model":<40} {"CER":>8} {"WER":>8}')
print('-' * 60)
print(f'{"EasyOCR (Arabic, off-the-shelf)":<40} {avg_cer:>8.4f} {avg_wer:>8.4f}')
print(f'{"Qwen2.5-VL-7B + LoRA (checkpoint-1120)":<40} {qwen["avg_cer"]:>8.4f} {qwen["avg_wer"]:>8.4f}')
print()
print('Note: TrOCR attempted but discarded — English-only tokenizer')
print('      incompatible with Arabic script. See Challenges section.')
print('\nStage 7 complete.')

=== Stage 7: Baseline Comparison ===

Model                                         CER      WER
------------------------------------------------------------
EasyOCR (Arabic, off-the-shelf)            0.4764   1.0582
Qwen2.5-VL-7B + LoRA (checkpoint-1120)     0.3283   0.6554

Note: TrOCR attempted but discarded — English-only tokenizer
      incompatible with Arabic script. See Challenges section.

Stage 7 complete.


## Step 8.8 — Qualitative analysis: sample predictions

Inspect what EasyOCR actually produced on the first 10 images. This gives
a human-readable sense of where it succeeds and where it fails, and feeds
directly into the error analysis discussion in the paper.

In [10]:
print('=== EasyOCR Sample Predictions vs GPT Reference ===\n')
for i, s in enumerate(sample_outputs):
    print(f'[{i+1}]')
    print(f'  REF : {s["reference"]}')
    print(f'  PRED: {s["predicted"]}')
    print()

=== EasyOCR Sample Predictions vs GPT Reference ===

[1]
  REF : بلفات اليمين القديمة.
  PRED: لمات الس»ن المد دمه

[2]
  REF : والجنة والأذان، كان لهم التقدم في خير الجماعة الحاصلة الإسلام، وقد
  PRED: = اللغهآ واوذب ء كان لهم اإدفصل كرآ حع ء اخبا العهلته ام7صله  بالاسلدم ( ,قد

[3]
  REF : ف إحصري المقصود التي نصف جبل نقرأ "على ضيم لنصر لوحات-
  PRED: غ إهرى لحصائرلتي لصفا جبل نقرا عان ضح لنصرلوحات

[4]
  REF : كطلوها كنو المستقبلا لتكون ثابتة ومشهورة وان لا نقبل كل جديد مكلن؟ نه حدي.
  PRED: خصوها خو السفبل لتكون  ابته وشمرد وان ( نقبل كا دد بس معان { ه ٧ .

[5]
  REF : لا يجري حولها والتعامل معه بتطوير أنظمتها وإدارة الرقابة ومن خلال استراتيجية
  PRED: ا _ هولها والتعامل مهه بهوير ٦ نخمعا رإدا ءت@ا < رم طلال اسوا تود

[6]
  REF : تم سلم كافة سلطات الحكم تقريبا
  PRED: تم سلم كاما سلطات الحم تقرببا

[7]
  REF : المخول لي؟ . أم الموظف المسؤول الإجراءات بأدب، ثم بجم ثم اخر.
  PRED: اللخول ى ؟" . أتم الموظف امسؤول الإجراءات لآدر جهع ( تم اخر

[8]
  REF : والكتابات المؤرخة بهذه الطريقة 

## Step 8.9 — Complete results summary across all stages

This cell loads the saved JSON files from Stages 6 and 7 and prints every
number needed for the paper tables. Run it at any point after NB_07 and
NB_08 have completed to get a consolidated view.

In [11]:
import json

print('=' * 60)
print('COMPLETE RESULTS SUMMARY — ALL STAGES')
print('=' * 60)

# Stage 6 — NER
with open(f'{PROJECT_ROOT}/logs/stage6/ner_results.json') as f:
    ner = json.load(f)
print('\n--- Stage 6: NER ---')
print(f'Transcriptions analyzed : {ner["num_transcriptions"]}')
print(f'GPT entities found      : {sum(ner["gpt_entity_counts"].values())}  {ner["gpt_entity_counts"]}')
print(f'Qwen entities found     : {sum(ner["qwen_entity_counts"].values())}  {ner["qwen_entity_counts"]}')
print(f'Precision               : {ner["comparison_metrics"]["precision"]}')
print(f'Recall                  : {ner["comparison_metrics"]["recall"]}')
print(f'F1                      : {ner["comparison_metrics"]["f1"]}')

# Stage 6 — POS
with open(f'{PROJECT_ROOT}/logs/stage6/pos_results.json') as f:
    pos = json.load(f)
print('\n--- Stage 6: POS Tagging ---')
print(f'Total tokens tagged     : {pos["total_tokens"]}')
print(f'Tag accuracy (Qwen/GPT) : {pos["pos_accuracy"] * 100:.2f}%')
print(f'GPT top tags  — noun: {pos["gpt_tag_counts"].get("noun",0)}, '
      f'noun_prop: {pos["gpt_tag_counts"].get("noun_prop",0)}, '
      f'verb: {pos["gpt_tag_counts"].get("verb",0)}, '
      f'prep: {pos["gpt_tag_counts"].get("prep",0)}')
print(f'Qwen top tags — noun: {pos["qwen_tag_counts"].get("noun",0)}, '
      f'noun_prop: {pos["qwen_tag_counts"].get("noun_prop",0)}, '
      f'verb: {pos["qwen_tag_counts"].get("verb",0)}, '
      f'prep: {pos["qwen_tag_counts"].get("prep",0)}')

# Stage 7 — EasyOCR baseline
with open(f'{PROJECT_ROOT}/logs/stage7/easyocr_results.json') as f:
    baseline = json.load(f)
print('\n--- Stage 7: Baseline Comparison ---')
print(f'Model           : {baseline["model"]}')
print(f'Architecture    : {baseline["architecture"]}')
print(f'Eval samples    : {baseline["num_eval_samples"]}')
print(f'CER             : {baseline["cer"]:.4f}')
print(f'WER             : {baseline["wer"]:.4f}')

# Qwen for comparison
with open(f'{PROJECT_ROOT}/logs/run-2/eval_results_checkpoint-1120.json') as f:
    qwen = json.load(f)
print('\n--- For comparison: Qwen2.5-VL-7B + LoRA (checkpoint-1120) ---')
print(f'CER             : {qwen["avg_cer"]:.4f}')
print(f'WER             : {qwen["avg_wer"]:.4f}')
print(f'Exact match     : {qwen["exact_match_rate"]}%')
print(f'Valid JSON      : {qwen["valid_json_rate"]}%')
print(f'Avg inference   : {qwen["avg_inference_time"]} s/sample')

print('\n--- TrOCR (attempted, discarded) ---')
print('Result          : Failed — English-only tokenizer incompatible with Arabic script')

print('\n' + '=' * 60)
print('All stages complete. Ready to write the paper.')
print('=' * 60)

COMPLETE RESULTS SUMMARY — ALL STAGES

--- Stage 6: NER ---
Transcriptions analyzed : 280
GPT entities found      : 132  {'MISC': 28, 'LOC': 56, 'PERS': 39, 'ORG': 9}
Qwen entities found     : 122  {'MISC': 24, 'LOC': 55, 'PERS': 31, 'ORG': 12}
Precision               : 0.2642
Recall                  : 0.2456
F1                      : 0.2545

--- Stage 6: POS Tagging ---
Total tokens tagged     : 3056
Tag accuracy (Qwen/GPT) : 20.18%
GPT top tags  — noun: 1077, noun_prop: 564, verb: 399, prep: 366
Qwen top tags — noun: 1005, noun_prop: 525, verb: 422, prep: 348

--- Stage 7: Baseline Comparison ---
Model           : EasyOCR (Arabic)
Architecture    : CRAFT detector + CRNN recognizer
Eval samples    : 280
CER             : 0.4764
WER             : 1.0582

--- For comparison: Qwen2.5-VL-7B + LoRA (checkpoint-1120) ---
CER             : 0.3283
WER             : 0.6554
Exact match     : 1.07%
Valid JSON      : 92.14%
Avg inference   : 3.83 s/sample

--- TrOCR (attempted, discarded) ---
Res

# final evaluation summary


**EasyOCR qualitative output (8.8)**

The sample predictions tell a clear story. EasyOCR struggles significantly with this dataset. Sample 6 is the closest to correct — "تم سلم كافة سلطات الحكم تقريبا" vs "تم سلم كاما سلطات الحم تقرببا" — a few character substitutions but the structure is preserved. Everything else shows severe degradation: numbers and symbols inserted mid-word (sample 2, 5), completely unrecognizable character sequences (sample 8, 9), and garbled word boundaries throughout. This is consistent with EasyOCR's CER of 0.476 — it is reading something, but handwritten Arabic paragraph-level images from AHTD are far harder than the printed or semi-printed text EasyOCR was designed for. The `paragraph=True` merging also struggles when text lines are not cleanly separated.

**Baseline comparison (8.7)**

| Model | CER | WER |
|---|---|---|
| Qwen2.5-VL-7B + LoRA (checkpoint-1120) | 0.3283 | 0.6554 |
| EasyOCR (Arabic, off-the-shelf) | 0.4764 | 1.0582 |
| TrOCR (discarded) | — | — |

Qwen outperforms EasyOCR on both metrics. CER is 31% lower (0.328 vs 0.476) and WER is 38% lower (0.655 vs 1.058). The WER comparison is particularly meaningful: EasyOCR's WER exceeding 1.0 means it is on average producing more words than the reference, which matches what you see in the samples — fragmented characters being read as separate tokens, symbols being inserted, lines being merged incorrectly. Qwen's WER of 0.655 is still high but reflects a model that at least generates coherent Arabic text of roughly the right length.

This is the core finding for Stage 7 in the paper: a knowledge-distilled 7B VLM fine-tuned on 1,120 samples outperforms an off-the-shelf Arabic OCR engine specifically designed for this task, with no additional engineering. That is a meaningful result.

**NER summary (8.9)**

The entity counts are close between GPT (132) and Qwen (122), with similar type distributions. The low F1 of 0.2545 reflects exact-string matching across two imperfect transcriptions, not a failure of entity detection, as discussed earlier. Worth noting for the paper: Qwen finds slightly fewer entities overall but maintains proportionally similar LOC/PERS/MISC/ORG ratios to GPT, which suggests the transcriptions are linguistically consistent even where they differ character-by-character.

**POS summary (8.9)**

The tag distributions are nearly identical — noun dominates at 35.2% (GPT) vs 33.9% (Qwen), noun_prop at 18.5% vs 17.7%, verb and prep within one percentage point. The 20.18% positional accuracy is misleading as discussed: it reflects the 88-token length mismatch between sequences, not tagging quality. The distribution similarity is the meaningful signal here, and it is strong.

**The overall picture**

Across all three evaluation dimensions — character-level transcription (CER/WER), named entity recognition, and morphological tagging — the fine-tuned Qwen adapter performs consistently and better than the classical baseline. The knowledge distillation approach worked: a student model trained on GPT-labeled handwriting data learned to read Arabic handwriting well enough to outperform purpose-built OCR software on the same task.